In [ ]:
# 📦 1. 环境准备与数据加载
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 通过 rc 参数设置中文字体(Seaborn 推荐做法)
sns.set_theme(
    style="whitegrid",
    font_scale=1.1,
    rc={
        "font.sans-serif": ["Microsoft YaHei"],
        "font.family": "sans-serif",
        "axes.unicode_minus": False
    }
)

# 加载 taxis 数据集:纽约出租车行程数据
df = sns.load_dataset("taxis")

# 快速查看数据结构
print(df.head())
print(f"\n数据规模:{df.shape}")
print(f"\n出发区域分布:")
print(df["pickup_borough"].value_counts())

In [ ]:
# 📦 2. 基础箱线图:按区域对比车费分布
fig, ax = plt.subplots(figsize=(10, 6))

# 过滤掉未知区域和空值
df_clean = df[df["pickup_borough"].notna() & (df["pickup_borough"] != "Unknown")]

# Seaborn 一行代码画出分组箱线图
sns.boxplot(
    data=df_clean,
    x="pickup_borough",
    y="fare",
    hue="pickup_borough",
    palette="Set2",
    legend=False,
    ax=ax
)

# Matplotlib 精细美化
ax.set_title("不同出发区域的出租车费对比:箱线图", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("出发区域", fontsize=12)
ax.set_ylabel("车费 (美元)", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# 🎻 3. 小提琴图:箱线图 + 核密度估计
fig, ax = plt.subplots(figsize=(10, 6))

# Seaborn 一行代码画出小提琴图
sns.violinplot(
    data=df_clean,
    x="pickup_borough",
    y="fare",
    hue="pickup_borough",
    palette="Set2",
    inner="box",
    legend=False,
    ax=ax
)

# Matplotlib 精细美化
ax.set_title("不同出发区域的出租车费对比:小提琴图", fontsize=16, fontweight="bold", pad=15)
ax.set_xlabel("出发区域", fontsize=12)
ax.set_ylabel("车费 (美元)", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# 🔍 4. 高级玩法:精准提取并标注异常值
fig, ax = plt.subplots(figsize=(12, 7))

# 先画小提琴图
sns.violinplot(
    data=df_clean,
    x="pickup_borough",
    y="fare",
    hue="pickup_borough",
    palette="pastel",
    inner="box",
    legend=False,
    ax=ax
)

# 提取 Manhattan 区域的异常值
manhattan = df_clean[df_clean["pickup_borough"] == "Manhattan"]
q1 = manhattan["fare"].quantile(0.25)
q3 = manhattan["fare"].quantile(0.75)
iqr = q3 - q1

# 计算异常值阈值
upper_bound = q3 + 1.5 * iqr
lower_bound = q1 - 1.5 * iqr

# 筛选异常值
outliers = manhattan[(manhattan["fare"] > upper_bound) | (manhattan["fare"] < lower_bound)]

# 在图上标注最高价异常值
top_outliers = outliers.nlargest(3, "fare")
for _, row in top_outliers.iterrows():
    ax.annotate(
        f"${row['fare']:.0f}",
        xy=(0, row["fare"]),
        xytext=(0.3, row["fare"]),
        fontsize=10,
        color="red",
        fontweight="bold",
        arrowprops=dict(arrowstyle="->", color="red", lw=1.5)
    )

ax.set_title("Manhattan 出发车费异常值分析(标注 Top 3 天价订单)", fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("出发区域", fontsize=12)
ax.set_ylabel("车费 (美元)", fontsize=12)

# 添加异常值阈值线
ax.axhline(upper_bound, color="orange", linestyle="--", linewidth=2, 
           label=f"异常值上限: ${upper_bound:.0f}")
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

# 打印异常值详细信息
print(f"\n🚨 Manhattan 区域共发现 {len(outliers)} 个异常订单")
print(f"异常值阈值:低于 ${lower_bound:.0f} 或高于 ${upper_bound:.0f}")
print("\nTop 3 天价订单详情:")
print(top_outliers[["pickup", "pickup_borough", "fare", "total", "distance"]])

In [ ]:
# 📅 5. 时间维度分析:工作日 vs 周末的车费差异
df_clean = df_clean.copy()
df_clean["day_of_week"] = pd.to_datetime(df_clean["pickup"]).dt.dayofweek
df_clean["is_weekend"] = df_clean["day_of_week"].isin([5, 6]).map({True: "周末", False: "工作日"})

# 过滤掉样本太少的区域
valid_boroughs = df_clean["pickup_borough"].value_counts()
valid_boroughs = valid_boroughs[valid_boroughs > 50].index
df_filtered = df_clean[df_clean["pickup_borough"].isin(valid_boroughs)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图:工作日
sns.boxplot(
    data=df_filtered[df_filtered["is_weekend"] == "工作日"],
    x="pickup_borough",
    y="fare",
    hue="pickup_borough",
    palette="Blues",
    legend=False,
    ax=axes[0]
)
axes[0].set_title("工作日:各区域车费分布", fontsize=14, fontweight="bold")
axes[0].set_xlabel("出发区域", fontsize=11)
axes[0].set_ylabel("车费 (美元)", fontsize=11)

# 右图:周末
sns.boxplot(
    data=df_filtered[df_filtered["is_weekend"] == "周末"],
    x="pickup_borough",
    y="fare",
    hue="pickup_borough",
    palette="Oranges",
    legend=False,
    ax=axes[1]
)
axes[1].set_title("周末:各区域车费分布", fontsize=14, fontweight="bold")
axes[1].set_xlabel("出发区域", fontsize=11)
axes[1].set_ylabel("")

plt.suptitle("工作日 vs 周末:纽约出租车费对比分析", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 🎨 6. 高级定制:自定义离群点样式与颜色
fig, ax = plt.subplots(figsize=(12, 8))

# 绘制小提琴图，隐藏内部元素以便叠加其他图层
sns.violinplot(
    data=df_filtered,
    x="pickup_borough",
    y="fare",
    hue="pickup_borough",
    palette="Set2",
    inner=None,  # 不显示内部元素，保持图形简洁
    legend=False,
    ax=ax
)

# 叠加箱线图展示四分位数
sns.boxplot(
    data=df_filtered,
    x="pickup_borough",
    y="fare",
    color="white",
    width=0.3,  # 箱子宽度较小
    showcaps=False,  # 不显示须线端点
    boxprops={'facecolor': 'none', 'edgecolor': 'black', 'linewidth': 1.5},
    whiskerprops={'linewidth': 1.5},
    medianprops={'linewidth': 2, 'color': 'black'},
    ax=ax
)

# 用 stripplot 叠加原始数据点(抽样显示)
sns.stripplot(
    data=df_filtered.sample(500),
    x="pickup_borough",
    y="fare",
    color="gray",
    alpha=0.4,
    size=4,
    ax=ax
)

# 用不同颜色高亮极端异常值(车费 > 100 美元)
extreme = df_filtered[df_filtered["fare"] > 100]
if len(extreme) > 0:
    sns.scatterplot(
        data=extreme,
        x="pickup_borough",
        y="fare",
        color="red",
        s=120,
        alpha=0.8,
        edgecolor="black",
        linewidth=1,
        label="极端异常值 (>$100)",
        ax=ax
    )

ax.set_title("纽约出租车费分布：小提琴图 + 箱线图 + 原始数据点 + 异常值高亮", 
             fontsize=15, fontweight="bold", pad=15)
ax.set_xlabel("出发区域", fontsize=12)
ax.set_ylabel("车费 (美元)", fontsize=12)
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig("taxi_fare_violin_advanced.png", dpi=300, bbox_inches="tight")
print("✅ 图表已保存为 'taxi_fare_violin_advanced.png'")
plt.show()

# 打印统计信息
print("\n📊 各区域车费统计：")
print(df_filtered.groupby("pickup_borough")["fare"].describe())

In [ ]:
# 💾 7. 完整代码模板(含高清导出)
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 1. 设置主题 + 中文支持
sns.set_theme(
    style="whitegrid",
    font_scale=1.1,
    rc={
        "font.sans-serif": ["Microsoft YaHei"],
        "font.family": "sans-serif",
        "axes.unicode_minus": False
    }
)

# 2. 加载数据(可替换为你自己的 CSV)
# df = pd.read_csv("your_data.csv")
df = sns.load_dataset("taxis")

# 3. 数据清洗
df_clean = df[df["pickup_borough"].notna() & (df["pickup_borough"] != "Unknown")]

# 4. 创建画布
fig, ax = plt.subplots(figsize=(10, 6))

# 5. Seaborn 画小提琴图
sns.violinplot(
    data=df_clean,
    x="pickup_borough",
    y="fare",
    hue="pickup_borough",
    palette="Set2",
    inner="box",
    legend=False,
    ax=ax
)

# 6. Matplotlib 精细美化
ax.set_title("各区域车费分布对比", fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("出发区域", fontsize=12)
ax.set_ylabel("车费 (美元)", fontsize=12)

plt.tight_layout()

# 7. 导出与显示
fig.savefig("boxplot_violin_comparison.png", dpi=300, bbox_inches="tight")
print("✅ 图表已保存为 'boxplot_violin_comparison.png'")
plt.show()